# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

In [1]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

True

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [2]:
LITE_MODE = True

In [3]:
username = "hackSameer"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

README.md:   0%|          | 0.00/737 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.06M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/995k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/989k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 10,000 items
title='Pacific Drums PDCM0708STSB 7 x 8 Inches Tom with Chrome Hardware - Silver to Black Fade' category='Musical_Instruments' price=199.99 full='Pacific Drums  7 x 8 Inches Tom with Chrome Hardware - Silver to Black Fade\n[\'Concept Series is a new line of PDP drums that are available in both; all-maple or all-birch shells. All shells are 7-ply, except for the snare which is 10-ply. Through our research with the DW Custom Shop, we have found some particular detail features that have a big effect on sound and quality. We believe every drummer deserves these features, so we included DW-style STM mounting systems, True Pitch tuning, F.A.S.T. tom sizing, proportional counter hoop sizing , and Remo drum heads. Snare drums come with the DW MAG throw-off and boutique-style copper wires. Bass drums come with die-cast claw hooks, and all drums feature the new retro-inspired dual-turret tube lug. Concept Maple drums are available in six specialty high-gloss finishes, all wit

In [4]:
items[2].id

In [5]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [6]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [7]:
print(items[0].full)

Pacific Drums  7 x 8 Inches Tom with Chrome Hardware - Silver to Black Fade
['Concept Series is a new line of PDP drums that are available in both; all-maple or all-birch shells. All shells are 7-ply, except for the snare which is 10-ply. Through our research with the DW Custom Shop, we have found some particular detail features that have a big effect on sound and quality. We believe every drummer deserves these features, so we included DW-style STM mounting systems, True Pitch tuning, F.A.S.T. tom sizing, proportional counter hoop sizing , and Remo drum heads. Snare drums come with the DW MAG throw-off and boutique-style copper wires. Bass drums come with die-cast claw hooks, and all drums feature the new retro-inspired dual-turret tube lug. Concept Maple drums are available in six specialty high-gloss finishes, all with chrome hardware, except for pearlescent black (black hardware). Concept Birch drums share all of the great features of Concept Maple, but with all-birch shells. They 

In [8]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


Title: Pacific Drums 7×8” Tom – Silver‑to‑Black Fade  
Category: Musical Instruments – Drums  
Brand: Pacific Drums  
Description: 7‑inch tom with premium copper and chrome hardware, fading from silver to black.  
Details: 8‑inch shell, 7‑lb weight, True‑Pitch tuning, dual‑turret lug, and STM mounting system.

Input tokens: 534
Output tokens: 98
Cost: 0.007 cents


In [24]:

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


### System:
Product: Pacific Drums 7 x 8 Inches Tom
Category: Electronics
Brand: Pacific Drums
Description: Premium electronic drum set with high-quality features and durable construction.
Details: Equipped with DW-style STM mounting systems for a responsive sound.

Input tokens: 495
Output tokens: 56
Cost: 0.000 cents


In [14]:
MODEL = "openai/gpt-oss-20b"


In [15]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [16]:
items[0]

<Pacific Drums PDCM0708STSB 7 x 8 Inches Tom with Chrome Hardware - Silver to Black Fade = $199.99>

In [17]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Pacific Drums  7 x 8 Inches Tom with Chrome Hardware - Silver to Black Fade\\n[\'Concept Series is a new line of PDP drums that are available in both; all-maple or all-birch shells. All shells are 7-ply, except for the snare which is 10-ply. Through our research with the DW Custom Shop, we have found some particular detail features that have a big effect on sound and quality. We believe every drummer deserves these features, so we included DW-style STM mounting systems, True Pitch tuning, F.A.S.T. tom sizing, proportional counter hoop sizing , and Re

In [18]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [19]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [20]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [ ]:

import json
from tqdm import tqdm
from litellm import completion

LOCAL_MODEL = "ollama/llama3.2"
OLLAMA_URL = "http://localhost:11434"

with open("jsonl/0_1000.jsonl", "r", encoding="utf-8") as f:
    lines = f.readlines()

output_file = "jsonl/batch_results_llama.jsonl"

with open(output_file, "w", encoding="utf-8") as f_out:
    for line in tqdm(lines):
        if line.strip():
            data = json.loads(line)
            custom_id = data["custom_id"]
            messages = data["body"]["messages"]
            try:
                res = completion(messages=messages, model=LOCAL_MODEL, api_base=OLLAMA_URL)
                
                result = {
                    "custom_id": custom_id,
                    "response": {"body": {"choices": [{"message": {"content": res.choices[0].message.content}}]}}
                }
                f_out.write(json.dumps(result) + "\n")
            except Exception as e:
                print(f"Error on ID {custom_id}: {e}")

print("All Data is now saved in SSD")


In [ ]:
file_id = response.id
file_id

In [ ]:
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

In [ ]:
result = groq.batches.retrieve(response.id)
result

In [ ]:
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [ ]:
with open("jsonl/batch_results_llama.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [ ]:
print(items[0].full)

In [ ]:
print(items[1000].summary)

## I've put exactly this logic into a Batch class

- Divides items into groups of 1,000
- Kicks off batches for each
- Allows us to monitor and collect the results when complete

## COSTS

Using Groq, for me - this cost under $1 for the Lite dataset and under $30 for the big dataset

But you don't need to pay anything! In the next lab, you can load my pre-processed results

In [ ]:
Batch.create(items, LITE_MODE)

In [ ]:
Batch.run()

In [ ]:
Batch.fetch()

In [ ]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
print(items[10234].summary)

In [ ]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)

In [ ]:
username = "ed-donner"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

## And here they are!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
